In [74]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import mysql.connector

In [75]:
from utils import(
    ejecutar_y_guardar
)

In [76]:
conn = mysql.connector.connect(
    host = "datamart.mex.internal.lyftbikes.com",
    port =3306,
    database= "mex_datawarehouse_bss4",
    user= "dm_mex_ge",
    password= "m+y#J9JI9F+^4qjSJLu^R",
)

cursor = conn.cursor()

In [77]:
def obtener_mes_anterior(fecha=None):
    if fecha is None:
        fecha = datetime.today()
    elif isinstance(fecha, str):
        fecha = datetime.strptimw(fecha, "%Y.%m.%d")

    inicio_mes_actual = fecha.replace(
        day=1,
        hour=0,
        minute=0,
        second=0,
        microsecond=0
    )

    inicio_mes_anterior = (
        inicio_mes_actual
        - relativedelta(months=1)
    )

    return(
         inicio_mes_anterior.strftime("%Y-%m-%d %H:%M:%S"),
        inicio_mes_actual.strftime("%Y-%m-%d %H:%M:%S")
    )

In [78]:
fecha_inicio, fecha_fin = obtener_mes_anterior()


In [79]:
fecha_fin

'2026-06-01 00:00:00'

In [80]:
fecha_inicio

'2026-05-01 00:00:00'

In [81]:
query_balaceo_orig =f"""
        SELECT 
    b.displayedNumber AS ID_bicicleta,
    CASE WHEN b.bikeModelName = 'CLASSIC' THEN 'Mecánica' ELSE NULL END AS t_bicicleta,
    t.id AS ID_empleado,
    DATE_FORMAT(CONVERT_TZ(FROM_UNIXTIME(i.actionDateInMs/1000, '%Y-%m-%d %H:%i:%s'), 'UTC', 'America/Costa_Rica' ), '%d, %m, %Y') AS fech_inicio,
    DATE_FORMAT(CONVERT_TZ(FROM_UNIXTIME(f.actionDateInMs/1000, '%Y-%m-%d %H:%i:%s'), 'UTC', 'America/Costa_Rica' ), '%d, %m, %Y') AS fech_fin,
    DATE_FORMAT(CONVERT_TZ(FROM_UNIXTIME(i.actionDateInMs/1000, '%Y-%m-%d %H:%i:%s'), 'UTC', 'America/Costa_Rica' ), '%H:%i:%s') AS ini_viaj,
    DATE_FORMAT(CONVERT_TZ(FROM_UNIXTIME(f.actionDateInMs/1000, '%Y-%m-%d %H:%i:%s'), 'UTC', 'America/Costa_Rica' ), '%H:%i:%s') AS fin_viaj,
    ce_ini.logicalTerminal AS ciclo_inici,
    ce_fin.logicalTerminal AS ciclo_fin,
    DATE_FORMAT(SEC_TO_TIME((f.actionDateInMs - i.actionDateInMs) / 1000), '%H:%i:%s') AS tiemp_viaj

    FROM (SELECT * FROM BikeTechnicianActionFact WHERE actionType_id = 1) AS i
    LEFT JOIN BikeTechnicianDim AS t ON i.tech_id = t.id
    LEFT JOIN BikeDim AS b ON i.bike_id = b.id
    LEFT JOIN (SELECT * FROM BikeTechnicianActionFact WHERE actionType_id = 0) AS f ON i.bikeTransferByTechnicianId = f.bikeTransferByTechnicianId
    LEFT JOIN BikeStationDim AS ce_ini ON  i.station_id = ce_ini.id
    LEFT JOIN BikeStationDim AS ce_fin ON  f.station_id = ce_fin.id

    WHERE i.actionDateInMs >= UNIX_TIMESTAMP(CONVERT_TZ('{fecha_inicio}', 'America/Costa_Rica', 'UTC')) * 1000
    AND i.actionDateInMs < UNIX_TIMESTAMP(CONVERT_TZ('{fecha_fin}', 'America/Costa_Rica', 'UTC')) * 1000
    AND f.actionDateInMs > i.actionDateInMs
    """

In [82]:
ruta_archivo_balanceo_orig = (
    rf"C:\Users\tsunami.martinez\Downloads\Reportes"
        rf"\balanceo_{fecha_inicio[:7]}.csv")

In [83]:
reporte_balanceo1 = ejecutar_y_guardar(query_balaceo_orig, conn, ruta_archivo_balanceo_orig )

Archivo guadado: C:\Users\tsunami.martinez\Downloads\Reportes\balanceo_2026-05.csv


c:\Users\tsunami.martinez\Documents\Consultas SQL\utils.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [84]:
ruta_archivo_balanceo_nueva = (
    rf"C:\Users\tsunami.martinez\Downloads\Reportes"
        rf"\balanceoNuevaQ_{fecha_inicio[:7]}.csv")

In [85]:
query_balanceo = f"""
SELECT
    b.displayedNumber AS ID_bicicleta,

    CASE
        WHEN b.bikeModelName = 'CLASSIC'
        THEN 'Mecánica'
    END AS t_bicicleta,

    t.id AS ID_empleado,

    DATE_FORMAT(
        CONVERT_TZ(
            FROM_UNIXTIME(i.actionDateInMs/1000),
            'UTC',
            'America/Costa_Rica'
        ),
        '%d/%m/%Y'
    ) AS fech_inicio,

    DATE_FORMAT(
        CONVERT_TZ(
            FROM_UNIXTIME(f.actionDateInMs/1000),
            'UTC',
            'America/Costa_Rica'
        ),
        '%d/%m/%Y'
    ) AS fech_fin,

    DATE_FORMAT(
        CONVERT_TZ(
            FROM_UNIXTIME(i.actionDateInMs/1000),
            'UTC',
            'America/Costa_Rica'
        ),
        '%H:%i:%s'
    ) AS ini_viaj,

    DATE_FORMAT(
        CONVERT_TZ(
            FROM_UNIXTIME(f.actionDateInMs/1000),
            'UTC',
            'America/Costa_Rica'
        ),
        '%H:%i:%s'
    ) AS fin_viaj,

    ce_ini.logicalTerminal AS ciclo_inici,
    ce_fin.logicalTerminal AS ciclo_fin,

    SEC_TO_TIME(
        (f.actionDateInMs - i.actionDateInMs) / 1000
    ) AS tiemp_viaj

FROM BikeTechnicianActionFact AS i

INNER JOIN BikeTechnicianActionFact AS f
    ON i.bikeTransferByTechnicianId = f.bikeTransferByTechnicianId
    AND f.actionType_id = 0
    AND f.actionDateInMs > i.actionDateInMs

LEFT JOIN BikeTechnicianDim AS t
    ON i.tech_id = t.id

LEFT JOIN BikeDim AS b
    ON i.bike_id = b.id

LEFT JOIN BikeStationDim AS ce_ini
    ON i.station_id = ce_ini.id

LEFT JOIN BikeStationDim AS ce_fin
    ON f.station_id = ce_fin.id

WHERE i.actionType_id = 1

AND i.actionDateInMs >=
    UNIX_TIMESTAMP(
        CONVERT_TZ(
            '{fecha_inicio}',
            'America/Costa_Rica',
            'UTC'
        )
    ) * 1000

AND i.actionDateInMs <
    UNIX_TIMESTAMP(
        CONVERT_TZ(
            '{fecha_fin}',
            'America/Costa_Rica',
            'UTC'
        )
    ) * 1000

ORDER BY i.actionDateInMs
"""

In [86]:
reporte_balanceo1 = ejecutar_y_guardar(query_balaceo_orig, conn, ruta_archivo_balanceo_nueva )

Archivo guadado: C:\Users\tsunami.martinez\Downloads\Reportes\balanceoNuevaQ_2026-05.csv


c:\Users\tsunami.martinez\Documents\Consultas SQL\utils.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)
